In [6]:
!git clone https://github.com/Anamika457/Neuro-Symbolic-AI-for-Suspicious-Bot-Detection.git

fatal: destination path 'Neuro-Symbolic-AI-for-Suspicious-Bot-Detection' already exists and is not an empty directory.


In [7]:
import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'

!pip install -q torch_geometric
!pip install -q pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html


Torch: 2.10.0+cu128 | CUDA available: True


In [8]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)


Using device: cuda


In [9]:
DATA_DIR = "/kaggle/input/datasets/judithannbenny/tnsgat-data"    
GRAPH_PT_PATH = os.path.join(DATA_DIR, "graph_data.pt")   # Path A: pre-built PyG object
USER_IDS_CSV  = os.path.join(DATA_DIR, "sample_user_ids.csv")  # Path B fallback (id,label,split)
FEATURES_CSV  = os.path.join(DATA_DIR, "sample_features.csv")  # Path B fallback
EDGES_CSV     = os.path.join(DATA_DIR, "sample_edges.csv")     # Path B fallback (src,dst,weight)

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

HIDDEN_DIM   = 64
HEADS        = 4
DROPOUT      = 0.3
LR           = 2e-4
WEIGHT_DECAY = 5e-4
EPOCHS       = 400
PATIENCE     = 150  # early stopping on val F1



In [10]:
def check_label_distribution(user_ids_csv, expected_human=850, expected_bot=150):
    if not os.path.exists(user_ids_csv):
        print(f"{user_ids_csv} not found yet — skip this check until Person 1's file is in place.")
        return
    ids_df = pd.read_csv(user_ids_csv)
    counts = ids_df['label'].astype(str).str.lower().value_counts()
    print("Label counts:")
    print(counts)
    total = counts.sum()
    print(f"\nTotal: {total} | Bot ratio: {counts.get('bot', 0) / total:.3f}")

    n_human = counts.get('human', 0) + counts.get('genuine', 0)
    n_bot = counts.get('bot', 0)
    if n_human != expected_human or n_bot != expected_bot:
        print(f"\n Expected {expected_human} human / {expected_bot} bot, got {n_human} / {n_bot}. "
              f"Check with Person 1 before training on this.")
    else:
        print("\nMatches the expected 850/150 split.")

    if 'split' in ids_df.columns:
        print("\nPer-split label breakdown:")
        print(pd.crosstab(ids_df['split'], ids_df['label']))

check_label_distribution(USER_IDS_CSV)


Label counts:
label
human    850
bot      150
Name: count, dtype: int64

Total: 1000 | Bot ratio: 0.150

Matches the expected 850/150 split.

Per-split label breakdown:
label  bot  human
split            
train  150    850


In [11]:
def load_from_csv(user_ids_csv, features_csv, edges_csv):
    ids_df = pd.read_csv(user_ids_csv)
    feat_df = pd.read_csv(features_csv)
    edges_df = pd.read_csv(edges_csv)

    id_to_idx = {uid: i for i, uid in enumerate(ids_df['id'])}
    feat_df = feat_df.set_index('id').loc[ids_df['id']].reset_index()
    feature_cols = [c for c in feat_df.columns if c not in ('id', 'label', 'split')]
    x = torch.tensor(feat_df[feature_cols].values, dtype=torch.float)

    src = edges_df['source'].map(id_to_idx).values
    dst = edges_df['target'].map(id_to_idx).values
    valid = (~pd.isna(src)) & (~pd.isna(dst))
    src, dst = src[valid].astype(int), dst[valid].astype(int)
    edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)

    w = edges_df.loc[valid, 'weight'].values.astype('float32')
    edge_attr = torch.tensor(w, dtype=torch.float).view(-1, 1)

    label_map = {'human': 0, 'genuine': 0, 'bot': 1}
    y_raw = ids_df['label'].astype(str).str.lower().map(label_map)
    assert y_raw.notna().all(), "Unrecognized label values — check label_map against actual label strings"
    y = torch.tensor(y_raw.values, dtype=torch.long)

    d = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    d.node_ids = ids_df['id'].tolist()
    d.split = ids_df['split'].tolist() if 'split' in ids_df.columns else None
    return d
data = None
if data is None:
    data = load_from_csv(USER_IDS_CSV, FEATURES_CSV, EDGES_CSV)
    print("Built graph from raw CSVs")
    print(data)


Built graph from raw CSVs
Data(x=[1000, 22], edge_index=[2, 488], edge_attr=[488, 1], y=[1000], node_ids=[1000], split=[1000])


In [12]:
num_nodes = data.num_nodes
num_edges = data.edge_index.size(1)
deg = torch.zeros(num_nodes, dtype=torch.long)
deg.scatter_add_(0, data.edge_index[0], torch.ones(data.edge_index.size(1), dtype=torch.long))
isolated = (deg == 0).sum().item()

print(f"Nodes: {num_nodes} | Edges: {num_edges} | Isolated nodes: {isolated}")
print("Class balance (0=genuine, 1=bot):", torch.bincount(data.y).tolist())

Nodes: 1000 | Edges: 488 | Isolated nodes: 812
Class balance (0=genuine, 1=bot): [850, 150]


In [13]:
def stratified_split(y, val_frac=0.15, test_frac=0.15, seed=SEED):
    rng = np.random.default_rng(seed)
    n = y.size(0)
    idx = np.arange(n)
    train_mask = torch.zeros(n, dtype=torch.bool)
    val_mask = torch.zeros(n, dtype=torch.bool)
    test_mask = torch.zeros(n, dtype=torch.bool)
    for c in torch.unique(y).tolist():
        c_idx = idx[y.numpy() == c]
        rng.shuffle(c_idx)
        n_val = int(len(c_idx) * val_frac)
        n_test = int(len(c_idx) * test_frac)
        val_mask[c_idx[:n_val]] = True
        test_mask[c_idx[n_val:n_val + n_test]] = True
        train_mask[c_idx[n_val + n_test:]] = True
    return train_mask, val_mask, test_mask

split_attr = getattr(data, 'split', None)
has_usable_split = (
    split_attr is not None
    and set(pd.unique(split_attr)) >= {'train', 'val', 'test'}
)

if has_usable_split:
    split_arr = np.array(split_attr)
    train_mask = torch.tensor(split_arr == 'train')
    val_mask = torch.tensor(split_arr == 'val')
    test_mask = torch.tensor(split_arr == 'test')
    print("Using split.csv-provided split")
else:
    if split_attr is not None:
        print(f"'split' column only contains {sorted(set(pd.unique(split_attr)))} — that's TwiBot-22's "
              f"original dataset split (which train/val/test portion these accounts were sampled FROM), "
              f"not a split for this ~1,000-account experiment. Falling back to our own stratified split.")
    train_mask, val_mask, test_mask = stratified_split(data.y)
    print("Using fallback stratified random split (70/15/15)")

print(f"Train: {train_mask.sum().item()} | Val: {val_mask.sum().item()} | Test: {test_mask.sum().item()}")


'split' column only contains ['train'] — that's TwiBot-22's original dataset split (which train/val/test portion these accounts were sampled FROM), not a split for this ~1,000-account experiment. Falling back to our own stratified split.
Using fallback stratified random split (70/15/15)
Train: 702 | Val: 149 | Test: 149


/tmp/ipykernel_58/300264207.py:21: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  and set(pd.unique(split_attr)) >= {'train', 'val', 'test'}
/tmp/ipykernel_58/300264207.py:32: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  print(f"'split' column only contains {sorted(set(pd.unique(split_attr)))} — that's TwiBot-22's "


In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_np = data.x.cpu().numpy()
scaler.fit(x_np[train_mask.cpu().numpy()])
data.x = torch.tensor(scaler.transform(x_np), dtype=torch.float).to(device)

In [15]:
class BotGAT(nn.Module):
    def __init__(self, in_dim, hidden_dim, heads, num_classes=2, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.gat1 = GATv2Conv(in_dim, hidden_dim, heads=heads, edge_dim=1, dropout=dropout)
        self.gat2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, concat=False, edge_dim=1, dropout=dropout)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, edge_attr):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.gat1(x, edge_index, edge_attr)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.gat2(x, edge_index, edge_attr)
        x = F.elu(x)
        return self.classifier(x)   # logits, shape [num_nodes, num_classes]

model = BotGAT(in_dim=data.x.size(1), hidden_dim=HIDDEN_DIM, heads=HEADS, dropout=DROPOUT).to(device)
data = data.to(device)
print(model)


BotGAT(
  (gat1): GATv2Conv(22, 64, heads=4)
  (gat2): GATv2Conv(256, 64, heads=1)
  (classifier): Linear(in_features=64, out_features=2, bias=True)
)


In [16]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
# class weights, in case bots are the minority class (they will be)
class_counts = torch.bincount(data.y[train_mask]).float()
class_weights = (class_counts.sum() / (class_counts.numel() * class_counts)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

def run_eval(mask):
    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index, data.edge_attr)
        preds = logits.argmax(dim=1)
        y_true = data.y[mask].cpu().numpy()
        y_pred = preds[mask].cpu().numpy()
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        loss = criterion(logits[mask], data.y[mask]).item()
    return loss, acc, f1

best_val_f1 = -1.0
best_epoch = -1
patience_counter = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    logits = model(data.x, data.edge_index, data.edge_attr)
    loss = criterion(logits[train_mask], data.y[train_mask])
    loss.backward()
    optimizer.step()

    val_loss, val_acc, val_f1 = run_eval(val_mask)
    history.append({'epoch': epoch, 'train_loss': loss.item(), 'val_loss': val_loss, 'val_acc': val_acc, 'val_f1': val_f1})
    scheduler.step(val_f1)
    if val_f1 > best_val_f1:
        best_val_f1, best_epoch = val_f1, epoch
        patience_counter = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch,
            'val_f1': val_f1,
            'val_acc': val_acc,
            'config': {'hidden_dim': HIDDEN_DIM, 'heads': HEADS, 'dropout': DROPOUT, 'in_dim': data.x.size(1)},
        }, os.path.join(CHECKPOINT_DIR, 'best_tgat_checkpoint.pt'))
    else:
        patience_counter += 1

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | train_loss {loss.item():.4f} | val_loss {val_loss:.4f} "
              f"| val_acc {val_acc:.4f} | val_f1 {val_f1:.4f} | best_f1 {best_val_f1:.4f} @ ep{best_epoch}")

    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)")
        break

pd.DataFrame(history).to_csv(os.path.join(CHECKPOINT_DIR, 'training_log.csv'), index=False)
print(f"\nBest val F1: {best_val_f1:.4f} at epoch {best_epoch}")


Epoch   1 | train_loss 0.7013 | val_loss 0.6974 | val_acc 0.2148 | val_f1 0.2096 | best_f1 0.2096 @ ep1
Epoch  10 | train_loss 0.6620 | val_loss 0.6482 | val_acc 0.3423 | val_f1 0.3399 | best_f1 0.3399 @ ep10
Epoch  20 | train_loss 0.6310 | val_loss 0.6101 | val_acc 0.6376 | val_f1 0.5702 | best_f1 0.5702 @ ep20
Epoch  30 | train_loss 0.6140 | val_loss 0.5855 | val_acc 0.7248 | val_f1 0.6347 | best_f1 0.6347 @ ep28
Epoch  40 | train_loss 0.6087 | val_loss 0.5685 | val_acc 0.7584 | val_f1 0.6639 | best_f1 0.6700 @ ep35
Epoch  50 | train_loss 0.6006 | val_loss 0.5545 | val_acc 0.7718 | val_f1 0.6761 | best_f1 0.6761 @ ep46
Epoch  60 | train_loss 0.5597 | val_loss 0.5449 | val_acc 0.8054 | val_f1 0.7081 | best_f1 0.7081 @ ep54
Epoch  70 | train_loss 0.5330 | val_loss 0.5386 | val_acc 0.7987 | val_f1 0.7015 | best_f1 0.7081 @ ep54
Epoch  80 | train_loss 0.5581 | val_loss 0.5354 | val_acc 0.7919 | val_f1 0.6950 | best_f1 0.7081 @ ep54
Epoch  90 | train_loss 0.5535 | val_loss 0.5341 | val_ac

In [17]:
ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'best_tgat_checkpoint.pt'), map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

with torch.no_grad():
    logits = model(data.x, data.edge_index, data.edge_attr)
    probs = F.softmax(logits, dim=1)
    preds = logits.argmax(dim=1)

y_true = data.y[test_mask].cpu().numpy()
y_pred = preds[test_mask].cpu().numpy()

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
cm = confusion_matrix(y_true, y_pred)

print(f"Test Accuracy:  {acc:.4f}")
print(f"Test Precision: {prec:.4f}")
print(f"Test Recall:    {rec:.4f}")
print(f"Test F1 (macro):{f1:.4f}")
print("Confusion matrix [rows=true 0/1, cols=pred 0/1]:")
print(cm)

with open(os.path.join(CHECKPOINT_DIR, 'test_metrics.json'), 'w') as f:
    json.dump({'accuracy': acc, 'precision': prec, 'recall': rec, 'f1_macro': f1,
               'confusion_matrix': cm.tolist()}, f, indent=2)

Test Accuracy:  0.8322
Test Precision: 0.7023
Test Recall:    0.7888
Test F1 (macro):0.7288
Confusion matrix [rows=true 0/1, cols=pred 0/1]:
[[108  19]
 [  6  16]]


In [ ]:
from sklearn.model_selection import StratifiedKFold

def train_one_fold(train_idx, test_idx, val_frac_of_train=0.15, epochs=EPOCHS, patience=PATIENCE, seed=SEED):
    # carve a val set out of this fold's train indices, stratified
    y_train_full = data.y[train_idx].cpu().numpy()
    rng = np.random.default_rng(seed)
    tr_idx, val_idx = [], []
    for c in np.unique(y_train_full):
        c_pos = train_idx[y_train_full == c]
        c_pos = c_pos[torch.randperm(len(c_pos), generator=torch.Generator().manual_seed(seed))]
        n_val = max(1, int(len(c_pos) * val_frac_of_train))
        val_idx.append(c_pos[:n_val])
        tr_idx.append(c_pos[n_val:])
    tr_idx = torch.cat(tr_idx)
    val_idx = torch.cat(val_idx)

    fold_model = BotGAT(in_dim=data.x.size(1), hidden_dim=HIDDEN_DIM, heads=HEADS, dropout=DROPOUT).to(device)
    fold_opt = torch.optim.Adam(fold_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    cw = torch.bincount(data.y[tr_idx]).float()
    cw = (cw.sum() / (cw.numel() * cw)).to(device)
    fold_crit = nn.CrossEntropyLoss(weight=cw)

    best_f1, best_state, patience_ctr = -1.0, None, 0
    for epoch in range(1, epochs + 1):
        fold_model.train()
        fold_opt.zero_grad()
        logits = fold_model(data.x, data.edge_index, data.edge_attr)
        loss = fold_crit(logits[tr_idx], data.y[tr_idx])
        loss.backward()
        fold_opt.step()

        fold_model.eval()
        with torch.no_grad():
            val_logits = fold_model(data.x, data.edge_index, data.edge_attr)
            val_pred = val_logits.argmax(dim=1)[val_idx].cpu().numpy()
            val_f1 = f1_score(data.y[val_idx].cpu().numpy(), val_pred, average='macro', zero_division=0)

        if val_f1 > best_f1:
            best_f1, patience_ctr = val_f1, 0
            best_state = {k: v.clone() for k, v in fold_model.state_dict().items()}
        else:
            patience_ctr += 1
        if patience_ctr >= patience:
            break

    fold_model.load_state_dict(best_state)
    fold_model.eval()
    with torch.no_grad():
        test_logits = fold_model(data.x, data.edge_index, data.edge_attr)
        test_pred = test_logits.argmax(dim=1)[test_idx].cpu().numpy()
    y_true = data.y[test_idx].cpu().numpy()
    acc = accuracy_score(y_true, test_pred)
    _, _, f1, _ = precision_recall_fscore_support(y_true, test_pred, average='macro', zero_division=0)
    return acc, f1

K = 5 
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
all_idx = np.arange(data.num_nodes)
y_np = data.y.cpu().numpy()

fold_accs, fold_f1s = [], []
for fold, (train_i, test_i) in enumerate(skf.split(all_idx, y_np), start=1):
    train_i = torch.tensor(train_i, dtype=torch.long)
    test_i = torch.tensor(test_i, dtype=torch.long)
    acc, f1 = train_one_fold(train_i, test_i)
    fold_accs.append(acc)
    fold_f1s.append(f1)
    print(f"Fold {fold}/{K} | test acc {acc:.4f} | test F1 (macro) {f1:.4f}")

print(f"\nAcross {K} folds: acc = {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f} "
      f"| F1 (macro) = {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")

pd.DataFrame({'fold': range(1, K+1), 'test_acc': fold_accs, 'test_f1_macro': fold_f1s}) \
    .to_csv(os.path.join(CHECKPOINT_DIR, 'kfold_results.csv'), index=False)


Fold 1/5 | test acc 0.8300 | test F1 (macro) 0.7392
Fold 2/5 | test acc 0.8550 | test F1 (macro) 0.7668
Fold 3/5 | test acc 0.8600 | test F1 (macro) 0.7576
Fold 4/5 | test acc 0.8100 | test F1 (macro) 0.7085
Fold 5/5 | test acc 0.7950 | test F1 (macro) 0.6827

Across 5 folds: acc = 0.8300 ± 0.0251 | F1 (macro) = 0.7310 ± 0.0313


In [21]:
node_ids = getattr(data, 'node_ids', list(range(data.num_nodes)))
export_df = pd.DataFrame({
    'user_id': node_ids,
    'y_true': data.y.cpu().numpy(),
    'p_bot_gnn': probs[:, 1].cpu().numpy(),
    'pred_gnn': preds.cpu().numpy(),
})
export_path = os.path.join(CHECKPOINT_DIR, 'gnn_predictions.csv')
export_df.to_csv(export_path, index=False)
print(f"Saved {export_path}")
export_df.head()


Saved /kaggle/working/checkpoints/gnn_predictions.csv


,user_id,y_true,p_bot_gnn,pred_gnn
0,u100626326,0,0.509078,1
1,u1480557467033608193,0,0.589655,1
2,u847110556431605763,0,0.379296,0
3,u975131159075196929,1,0.606434,1
4,u84340979,0,0.350264,0


In [22]:
import os

print(os.listdir("/kaggle/working/checkpoints"))

['gnn_predictions.csv', 'training_log.csv', 'kfold_results.csv', 'test_metrics.json', 'best_tgat_checkpoint.pt']


In [ ]:
import shutil


shutil.make_archive(
    "/kaggle/working/checkpoints",  # output: checkpoints.zip
    "zip",
    "/kaggle/working/checkpoints"
)

print("Created: /kaggle/working/checkpoints.zip")

Created: /kaggle/working/checkpoints.zip
